# Drone env — browser eval

Pick a checkpoint, click **Build & load**. The recipe `just build-web MODEL=<path>` bakes those weights into a WASM bundle and the iframe below reloads against it.

In [ ]:
import subprocess, atexit, time, pathlib, glob, ipywidgets as W
from IPython.display import IFrame, display, clear_output

PROJECT = pathlib.Path('/work')
BUILD   = PROJECT / 'build' / 'web'
PORT    = 8765

BUILD.mkdir(parents=True, exist_ok=True)
srv = subprocess.Popen(
    ['python3', '-m', 'http.server', '-d', str(BUILD), str(PORT)],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
atexit.register(srv.terminate)
time.sleep(0.3)

def list_checkpoints():
    return sorted(glob.glob(str(PROJECT / 'checkpoints/drone/*.bin')))

def rebuild(model_path):
    return subprocess.run(
        ['just', 'build-web', f'MODEL={model_path}'],
        cwd=PROJECT, capture_output=True, text=True,
    )

print(f'Server up on :{PORT}')

In [ ]:
ckpts = list_checkpoints()
if not ckpts:
    print('No checkpoints in checkpoints/drone/. Train one first: just train cpu hover')
else:
    picker = W.Dropdown(options=ckpts, value=ckpts[-1], description='Model:', layout=W.Layout(width='80%'))
    button = W.Button(description='Build & load', button_style='primary')
    status = W.Output()
    frame  = W.Output()

    def on_click(_):
        with status:
            clear_output(); print(f'Building with {picker.value} ...')
        res = rebuild(picker.value)
        with status:
            clear_output()
            if res.returncode != 0:
                print('Build failed:\n' + res.stderr); return
            print(f'Loaded: {picker.value}')
        with frame:
            clear_output()
            display(IFrame(f'/proxy/{PORT}/game.html?v={int(time.time())}', width=960, height=640))

    button.on_click(on_click)
    display(W.VBox([W.HBox([picker, button]), status, frame]))
    on_click(None)